# PPE Detection (K3 Industri) — Notebook Google Colab
**Kandidat:** Muhammad Syauqi Fittuqo · Tugas 25 Nov 2026 · Deadline 26 Nov 2026
Kelas: `person, hat, gloves, vest, no_hat, no_gloves, no_vest` · Model: YOLOv8n (bobot awal COCO)

## Persiapan
1. **Runtime → Change runtime type → T4 GPU.**
2. Taruh `dataset_yolo.zip` di Google Drive pada `MyDrive/PPE_NBA-inital assignment/image-annotation/` (path ada di sel konfigurasi, `DRIVE_DIR`). Saat diminta, izinkan Colab mengakses Drive.
3. Jalankan sel dari atas ke bawah. Di akhir, hasil (`best.pt`, grafik, metrik) dikemas menjadi `ppe_outputs.zip` dan diunduh ke komputer.

Bila `test_media.zip` ditaruh di folder yang sama, itu dipakai untuk uji; kalau tidak, notebook meminta unggahan manual.

## 0. Setup

In [1]:
%pip -q install ultralytics
import os, sys, json, random, hashlib, time, csv, shutil, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch, yaml, cv2
from PIL import Image
import ultralytics
from ultralytics import YOLO

IN_COLAB = 'google.colab' in sys.modules
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
CLASSES = ['person','hat','gloves','vest','no_hat','no_gloves','no_vest']

DRIVE_DIR = Path('/content/drive/MyDrive/PPE_NBA-inital assignment/image-annotation')
BASE_WEIGHTS = 'coco'
EPOCHS, IMGSZ, BATCH, PATIENCE = 100, 640, 16, 30
VID_STEP = 2
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
WORKERS = 2 if IN_COLAB else 8
RUN_NAME = f"ppe_yolov8n_{BASE_WEIGHTS if BASE_WEIGHTS in ('coco','keremberke') else 'custom'}"

WORK = Path('/content/work') if IN_COLAB else Path.cwd()/'colab_work'
WORK.mkdir(parents=True, exist_ok=True)
ASSETS = WORK/'report_assets'
ASSETS.mkdir(exist_ok=True)
OUT = WORK/'outputs'/RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| Colab:', IN_COLAB)
print('device:', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU -> aktifkan GPU di Runtime!)')
print(dict(base=BASE_WEIGHTS, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED))

Note: you may need to restart the kernel to use updated packages.


ultralytics 8.4.218 | torch 2.15.0+cpu | Colab: False
device: cpu (CPU -> aktifkan GPU di Runtime!)
{'base': 'coco', 'epochs': 100, 'imgsz': 640, 'batch': 16, 'patience': 30, 'seed': 42}


## 1. Ambil dataset dari Google Drive dan siapkan

In [2]:
def load_from_drive(src, dest):
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    assert (src/'dataset_yolo.zip').exists(), f'dataset_yolo.zip tidak ditemukan di {src}'
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True)
    for name in ('dataset_yolo.zip', 'test_media.zip'):
        if (src/name).exists():
            zipfile.ZipFile(src/name).extractall(dest/Path(name).stem)
    return dest

In [3]:
def find_root(base):
    for p in [base, *sorted(base.rglob('*'))]:
        if p.is_dir() and (p/'images'/'train').is_dir() and (p/'labels'/'train').is_dir() and (p/'images'/'val').is_dir():
            return p
    return None

In [4]:
def prepare_dataset(src_root, ds):
    src = find_root(src_root)
    assert src is not None, 'struktur dataset tidak dikenali (butuh images/{train,val} dan labels/{train,val})'
    shutil.rmtree(ds, ignore_errors=True)
    shutil.copytree(src, ds, ignore=shutil.ignore_patterns('preview', 'crops', 'labels_prelabel', 'annotations', '*.zip'))
    (ds/'data.yaml').write_text(yaml.safe_dump({'path': str(ds), 'train': 'images/train', 'val': 'images/val',
                                                'names': {i: c for i, c in enumerate(CLASSES)}}, sort_keys=False))
    return ds

In [ ]:
DL = load_from_drive(DRIVE_DIR, WORK/'drive_download')
DS = prepare_dataset(DL, WORK/'dataset_yolo')
print('Dataset siap di', DS)
print((DS/'data.yaml').read_text())

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png'}

def listdir(p):
    return sorted(x for x in p.iterdir() if x.suffix.lower() in IMG_EXT)

In [ ]:
def count(split, files):
    c = np.zeros(7, int)
    bad = []
    for p in files:
        f = DS/'labels'/split/f'{p.stem}.txt'
        if not f.exists():
            bad.append(f'{p.name}: label tidak ada')
            continue
        for i, l in enumerate(x for x in f.read_text().split('\n') if x.strip()):
            t = l.split()
            try:
                k = int(t[0])
                v = list(map(float, t[1:5]))
                assert len(t) == 5 and 0 <= k < 7 and all(0 <= x <= 1 for x in v)
                c[k] += 1
            except Exception:
                bad.append(f'{p.name} baris {i+1}: {l}')
    return c, bad

In [ ]:
def plot_class_distribution(ctr, cva):
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    x = np.arange(7)
    ax.bar(x-0.2, ctr, 0.4, label='train')
    ax.bar(x+0.2, cva, 0.4, label='val')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES, rotation=20)
    ax.set_ylabel('jumlah instance')
    ax.set_title('Distribusi kelas')
    ax.legend()
    for i in range(7):
        ax.text(i-0.2, ctr[i]+2, str(ctr[i]), ha='center', fontsize=8)
        ax.text(i+0.2, cva[i]+2, str(cva[i]), ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(ASSETS/'class_distribution.png', dpi=150)
    plt.show()

In [ ]:
def md5(p):
    return hashlib.md5(p.read_bytes()).hexdigest()

In [ ]:
def dhash(p):
    px = list(Image.open(p).convert('L').resize((9, 8)).getdata())
    return sum(1 << (r*8+c) for r in range(8) for c in range(8) if px[r*9+c] > px[r*9+c+1])

In [ ]:
def check_no_leakage(tr, va):
    assert not ({p.name for p in tr} & {p.name for p in va}), 'nama sama di train & val'
    assert not ({md5(p) for p in tr} & {md5(p) for p in va}), 'isi identik di train & val'
    htr = [dhash(p) for p in tr]
    near = sum(1 for p in va if any(bin(dhash(p) ^ h).count('1') <= 5 for h in htr))
    print(f'AC-1 OK: nama sama=0, byte-identik=0, near-duplicate val~train={near}')
    return near

In [ ]:
tr, va = listdir(DS/'images/train'), listdir(DS/'images/val')
print(f'train={len(tr)} | val={len(va)}')

ctr, b1 = count('train', tr)
cva, b2 = count('val', va)
assert not (b1 + b2), f'label bermasalah: {(b1+b2)[:5]}'
print('Label valid.')
print(f"{'kelas':10s}{'train':>7s}{'val':>6s}")
for i, k in enumerate(CLASSES):
    print(f'{k:10s}{ctr[i]:7d}{cva[i]:6d}')

plot_class_distribution(ctr, cva)
near = check_no_leakage(tr, va)

## 2. Training dari bobot pretrained — AC-2, AC-3

In [ ]:
def resolve_start_weights(base):
    if base == 'coco':
        return 'yolov8n.pt'
    if base == 'keremberke':
        from huggingface_hub import list_repo_files, hf_hub_download
        repo = 'keremberke/yolov8n-protective-equipment-detection'
        return hf_hub_download(repo, [f for f in list_repo_files(repo) if f.endswith('.pt')][0])
    return base

In [ ]:
def train_model(start, name=RUN_NAME):
    model = YOLO(start)
    t0 = time.time()
    res = model.train(data=str(DS/'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED,
                      project=str(WORK/'runs'), name=name, exist_ok=True, pretrained=True, plots=True,
                      device=DEVICE, workers=WORKERS, verbose=False)
    return Path(res.save_dir), time.time() - t0

In [ ]:
START = resolve_start_weights(BASE_WEIGHTS)
print('Bobot awal:', START)
RUN, TRAIN_SEC = train_model(START)
print(f'selesai dalam {TRAIN_SEC/60:.1f} menit | run dir: {RUN}')
print(sorted(os.listdir(RUN/'weights')))

In [ ]:
def read_training_log(run):
    rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(open(run/'results.csv'))]
    best_ep = max(range(len(rows)), key=lambda i: float(rows[i]['metrics/mAP50-95(B)'])) + 1
    return rows, best_ep

In [ ]:
def show_training_summary(run, rows, best_ep):
    print('epoch dijalankan:', len(rows), '| epoch terbaik (mAP50-95):', best_ep)
    for k in ['train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
        print(f'{k:24s} awal={float(rows[0][k]):.4f}  akhir={float(rows[-1][k]):.4f}')
    plt.figure(figsize=(12, 6))
    plt.imshow(plt.imread(run/'results.png'))
    plt.axis('off')
    plt.title('Grafik loss dan metrik training')
    plt.show()
    shutil.copy(run/'results.png', ASSETS/'results.png')

In [ ]:
rows, best_ep = read_training_log(RUN)
show_training_summary(RUN, rows, best_ep)

## 3. Evaluasi data val — AC-4, AC-5

In [ ]:
def evaluate_model(weights, cva, name=RUN_NAME):
    best = YOLO(weights)
    m = best.val(data=str(DS/'data.yaml'), split='val', imgsz=IMGSZ, plots=True, device=DEVICE, workers=WORKERS,
                 project=str(WORK/'runs'), name=name+'_eval', exist_ok=True, verbose=False)
    overall = {'mAP50': float(m.box.map50), 'mAP50-95': float(m.box.map),
               'Precision': float(m.box.mp), 'Recall': float(m.box.mr)}
    per = [dict(kelas=CLASSES[ci], P=float(m.box.p[j]), R=float(m.box.r[j]),
                AP50=float(m.box.ap50[j]), AP5095=float(m.box.ap[j]), n_val=int(cva[ci]))
           for j, ci in enumerate(m.box.ap_class_index)]
    return best, overall, per, Path(m.save_dir), float(m.speed['inference'])

In [ ]:
def show_evaluation(overall, per, eval_dir):
    print({k: round(v, 4) for k, v in overall.items()})
    print(f"{'kelas':10s}{'P':>8s}{'R':>8s}{'AP50':>8s}{'AP50-95':>9s}{'n_val':>7s}")
    for r in sorted(per, key=lambda r: -r['AP50']):
        print(f"{r['kelas']:10s}{r['P']:8.3f}{r['R']:8.3f}{r['AP50']:8.3f}{r['AP5095']:9.3f}{r['n_val']:7d}")
    pairs = [('BoxP_curve.png', 'P_curve.png'), ('BoxR_curve.png', 'R_curve.png'),
             ('BoxPR_curve.png', 'PR_curve.png'), ('BoxF1_curve.png', 'F1_curve.png'),
             ('P_curve.png', 'P_curve.png'), ('R_curve.png', 'R_curve.png'), ('PR_curve.png', 'PR_curve.png'),
             ('confusion_matrix.png', 'confusion_matrix.png'),
             ('confusion_matrix_normalized.png', 'confusion_matrix_normalized.png')]
    for src, dst in pairs:
        if (eval_dir/src).exists():
            shutil.copy(eval_dir/src, ASSETS/dst)
    plt.figure(figsize=(8, 7))
    plt.imshow(plt.imread(ASSETS/'confusion_matrix_normalized.png'))
    plt.axis('off')
    plt.show()

In [ ]:
def check_ac5(overall):
    ac5 = (overall['mAP50'] > 0.85, overall['mAP50-95'] > 0.75)
    print('AC-5  mAP50 > 0.85  :', ac5[0], f"({overall['mAP50']:.3f})")
    print('AC-5  mAP50-95 > 0.75:', ac5[1], f"({overall['mAP50-95']:.3f})")
    return ac5

In [ ]:
best, overall, per, EV, SPEED = evaluate_model(RUN/'weights'/'best.pt', cva)
show_evaluation(overall, per, EV)
AC5 = check_ac5(overall)

## 4. Uji pada gambar dan video baru — AC-6
Gunakan gambar/video yang **bukan** bagian dari dataset. Sumber: `testing-file/` (atau `test_media/`) dari `test_media.zip` di folder Drive, atau unggah manual.

In [ ]:
VIDEO_EXT = ('.mp4', '.avi', '.mov', '.ogv', '.webm')

def collect_test_media(download_dir, tm):
    shutil.rmtree(tm, ignore_errors=True)
    (tm/'images').mkdir(parents=True)
    (tm/'video').mkdir()
    base = next((p for name in ('testing-file', 'test_media') for p in sorted(download_dir.rglob(name)) if p.is_dir()), None)
    if base is not None:
        for p in base.rglob('*'):
            if p.suffix.lower() in IMG_EXT:
                shutil.copy(p, tm/'images'/p.name)
            elif p.suffix.lower() in VIDEO_EXT:
                shutil.copy(p, tm/'video'/p.name)
    elif IN_COLAB:
        from google.colab import files
        print('testing-file tidak ada di folder Drive -> unggah >=5 gambar (dan/atau 1 video):')
        for n in files.upload():
            kind = 'video' if n.lower().endswith(VIDEO_EXT) else 'images'
            shutil.move(n, tm/kind/n)
    imgs = sorted(p for p in (tm/'images').iterdir() if p.suffix.lower() in IMG_EXT)
    vids = sorted((tm/'video').iterdir())
    return imgs, vids

In [ ]:
imgs, vids = collect_test_media(DL, WORK/'test_media')
print(f'gambar uji: {len(imgs)} | video uji: {len(vids)}')
assert len(imgs) >= 5 or vids, 'butuh >=5 gambar atau 1 video'

In [ ]:
def predict_images(model, imgs, conf=0.25):
    outs, det = [], {}
    for p in imgs:
        r = model.predict(str(p), conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        out = ASSETS/f'pred_{p.stem}.jpg'
        Image.fromarray(r.plot(line_width=2, font_size=10)[:, :, ::-1]).save(out, quality=88)
        outs.append(out)
        names = [CLASSES[int(c)] for c in r.boxes.cls]
        det[p.name] = {k: names.count(k) for k in sorted(set(names))}
        print(p.name, '->', len(names), 'deteksi', det[p.name])
    return outs, det

In [ ]:
def show_prediction_grid(outs):
    cols = 3
    rws = (len(outs) + cols - 1) // cols
    fig, axs = plt.subplots(rws, cols, figsize=(16, 4.6*rws))
    axs = np.atleast_1d(axs).ravel()
    for a, o in zip(axs, outs):
        a.imshow(Image.open(o))
        a.axis('off')
        a.set_title(o.stem[5:], fontsize=9)
    for a in axs[len(outs):]:
        a.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
outs, det_img = predict_images(best, imgs)
show_prediction_grid(outs)

In [ ]:
def detect_video(model, video_path, step=2, out_size=(960, 540), conf=0.25):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ow, oh = out_size
    writer = cv2.VideoWriter(str(ASSETS/'video_detect.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), fps/step, (ow, oh))
    totals = {k: 0 for k in CLASSES}
    frames_with, done, samples = 0, 0, []
    t0 = time.time()
    for fi in range(0, n, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, fr = cap.read()
        if not ok:
            break
        r = model.predict(fr, conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        pl = cv2.resize(r.plot(line_width=2, font_size=10), (ow, oh))
        writer.write(pl)
        done += 1
        frames_with += int(len(r.boxes) > 0)
        for c in r.boxes.cls:
            totals[CLASSES[int(c)]] += 1
        if done in (5, max(6, n//step//2), max(7, n//step - 5)):
            samples.append((fi, pl))
    writer.release()
    cap.release()
    print(f'video {Path(video_path).name}: {n} frame @ {fps:.0f} fps; diproses {done} frame dalam {time.time()-t0:.0f} dtk')
    print('frame dengan deteksi:', frames_with, '/', done)
    print('total deteksi per kelas:', totals)
    info = dict(name=Path(video_path).name, frames=n, fps=fps, processed=done, frames_with_det=frames_with, totals=totals)
    return info, samples

In [ ]:
def show_video_samples(samples):
    for k, (fi, pl) in enumerate(samples):
        cv2.imwrite(str(ASSETS/f'video_frame_{k}.jpg'), pl)
    fig, axs = plt.subplots(1, len(samples), figsize=(15, 3.6))
    for a, (fi, pl) in zip(np.atleast_1d(axs), samples):
        a.imshow(pl[:, :, ::-1])
        a.axis('off')
        a.set_title(f'frame {fi}', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
vid_info = None
if vids:
    vid_info, samples = detect_video(best, vids[0], step=VID_STEP)
    show_video_samples(samples)
else:
    print('tidak ada video; AC-6 dipenuhi lewat gambar saja')

## 5. Bonus — bandingkan YOLOv8n dan YOLOv8s
Kedua model dilatih dari bobot pretrained COCO pada dataset, split dan pengaturan yang sama (epoch, imgsz, batch, patience, seed). Hanya ukuran model yang berbeda. Hasil YOLOv8n diambil dari training di atas, jadi hanya YOLOv8s yang dilatih ulang (di T4 sekitar 10–20 menit).

In [ ]:
def count_params(weights):
    model = YOLO(weights)
    return sum(p.numel() for p in model.model.parameters())

In [ ]:
def make_row(model_name, run, train_sec, overall, rows, best_ep, speed):
    weights = run/'weights'/'best.pt'
    return dict(model=model_name, params_juta=round(count_params(weights)/1e6, 2),
                ukuran_mb=round(weights.stat().st_size/1e6, 1), epoch_dijalankan=len(rows), epoch_terbaik=best_ep,
                menit_training=round(train_sec/60, 1), inferensi_ms=round(speed, 1),
                mAP50=round(overall['mAP50'], 4), mAP50_95=round(overall['mAP50-95'], 4),
                precision=round(overall['Precision'], 4), recall=round(overall['Recall'], 4))

In [ ]:
def run_experiment(model_name):
    name = f'ppe_{model_name}_coco'
    run, train_sec = train_model(f'{model_name}.pt', name)
    rows_m, best_ep_m = read_training_log(run)
    _, overall_m, _, _, speed_m = evaluate_model(run/'weights'/'best.pt', cva, name)
    return make_row(model_name, run, train_sec, overall_m, rows_m, best_ep_m, speed_m)

In [ ]:
def save_comparison(results):
    cols = list(results[0].keys())
    with open(ASSETS/'model_comparison.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(results)
    print(''.join(f'{c:>18s}' for c in cols))
    for r in results:
        print(''.join(f'{str(r[c]):>18s}' for c in cols))

In [ ]:
def plot_comparison(results):
    metrics = ['mAP50', 'mAP50_95', 'precision', 'recall']
    x = np.arange(len(metrics))
    width = 0.8 / len(results)
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    for i, r in enumerate(results):
        vals = [r[m] for m in metrics]
        ax.bar(x + i*width - 0.4 + width/2, vals, width, label=r['model'])
        for xi, v in zip(x, vals):
            ax.text(xi + i*width - 0.4 + width/2, v + 0.01, f'{v:.2f}', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(['mAP50', 'mAP50-95', 'Precision', 'Recall'])
    ax.set_ylim(0, 1.05)
    ax.set_title('Perbandingan model pada data val')
    ax.legend()
    plt.tight_layout()
    plt.savefig(ASSETS/'model_comparison.png', dpi=150)
    plt.show()

In [ ]:
def comparison_verdict(results):
    top = max(results, key=lambda r: r['mAP50_95'])
    low = min(results, key=lambda r: r['mAP50_95'])
    print(f"mAP50-95 tertinggi: {top['model']} ({top['mAP50_95']:.3f}) vs {low['model']} ({low['mAP50_95']:.3f}), selisih {top['mAP50_95'] - low['mAP50_95']:.3f}")
    print(f"{top['model']}: {top['params_juta']} juta parameter, {top['menit_training']} menit training, {top['inferensi_ms']} ms/gambar")
    print('Catatan: val hanya', len(va), 'gambar dan satu seed, jadi selisih kecil bisa berasal dari variasi acak.')

In [ ]:
assert BASE_WEIGHTS == 'coco', 'perbandingan memakai bobot COCO untuk kedua model'
comparison = [make_row('yolov8n', RUN, TRAIN_SEC, overall, rows, best_ep, SPEED), run_experiment('yolov8s')]
save_comparison(comparison)
plot_comparison(comparison)
comparison_verdict(comparison)

## 6. Ringkasan untuk laporan dan unduh hasil

In [ ]:
def build_summary():
    return dict(
        kandidat='Muhammad Syauqi Fittuqo', model='yolov8n', base=BASE_WEIGHTS, start_weights=Path(START).name,
        epochs_set=EPOCHS, epochs_run=len(rows), best_epoch=best_ep, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED,
        device=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'), train_minutes=round(TRAIN_SEC/60, 1),
        n_train=len(tr), n_val=len(va), n_unlabeled=0,
        train_instances=dict(zip(CLASSES, map(int, ctr))), val_instances=dict(zip(CLASSES, map(int, cva))),
        overall=overall, per_class=per, ac5=dict(mAP50=bool(AC5[0]), mAP50_95=bool(AC5[1])), near_dup=int(near),
        test_images=det_img, video=vid_info, comparison=comparison,
        losses={k: [float(rows[0][k]), float(rows[-1][k])] for k in ['train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss']})

In [ ]:
def save_outputs(summary):
    json.dump(summary, open(ASSETS/'report_data.json', 'w'), indent=1)
    shutil.copy(RUN/'results.csv', ASSETS/'results.csv')
    shutil.copy(RUN/'args.yaml', ASSETS/'args.yaml')
    dest = OUT/'report_assets'
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(ASSETS, dest)
    shutil.copy(RUN/'weights/best.pt', OUT/'best.pt')
    shutil.copy(DS/'data.yaml', OUT/'data.yaml')
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            print(f'  {str(p.relative_to(OUT)):50s}{p.stat().st_size/1024:10.0f} KB')
    return Path(shutil.make_archive(str(WORK/'ppe_outputs'), 'zip', OUT))

In [ ]:
summary = build_summary()
ZIP = save_outputs(summary)
print('Arsip hasil:', ZIP)
print(json.dumps({k: summary[k] for k in ['overall', 'epochs_run', 'best_epoch', 'train_minutes', 'ac5']}, indent=1))

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

### Setelah selesai
- `ppe_outputs.zip` berisi `best.pt` (YOLOv8n), `data.yaml` dan folder `report_assets/` (grafik, prediksi, `report_data.json`, `model_comparison.csv` dan `model_comparison.png` untuk bonus).
- Sesi Colab bersifat sementara: unduh arsip sebelum sesi ditutup.
- Jangan lupa tulis di laporan: pengaturan (epoch, imgsz, batch), perangkat (GPU Colab), dan bahwa label dibuat dengan bantuan AI.